<center><img src="./img/pong-thumbnail.png" width="200" alt="Skills Network Logo"  /></center>
  


## Test Contenedores levantados:


## Dockers:

Para asegurarnos de que todos los servicios levantados en los contenedores funcionan correctamente, podemos definir algunos test automáticos y manuales para verificar cada servicio. A continuación están los pasos y comandos que hemos incluido para testear los contenedores y garantizar que todo está funcionando como se espera:

- ## 1. Verificar el estado de los contenedores
Después de ejecutar `make`, puedos usar este comando para verificar que todos los contenedores estén en ejecución y sin errores:

In [ ]:
make ps
docker compose -f ./src/docker-compose.yml ps
docker ps -a

Asegurarse de que todos los contenedores estén en estado "Up". Si hay alguno en estado "Exited" o "Restarting", eso indicaría un problema con ese servicio.

- ## 2. Comprobación de los logs:
Puedes revisar los logs de cada servicio para ver si se están generando errores o advertencias que no deberían estar allí. Esto es útil para depurar rápidamente problemas de configuración o dependencias faltantes:

In [ ]:
make logs
docker compose -f ./src/docker-compose.yml logs

Podemos filtrar los logs por servicio específico:

In [ ]:
make logs_service
Por favor, especifica un servicio. Uso: make logs_service SERVICE=<nombre_del_servicio>
make logs_service SERVICE=<nombre_del_servicio>
docker compose -f ./src/docker-compose.yml logs <service_name>

- ## 3. Tests específicos por servicio
    - ## a. SQLite (Base de datos):

    Una vez que el contenedor esté levantado, podemos ejecutar comandos SQL para verificar si la base de datos funciona correctamente. Por ejemplo, podemos conectarnos al contenedor de SQLite y listar las tablas para asegurarte de que está configurado:
    - **Test:** Verificar que SQLite está en ejecución
    
- **Comandos:** 

In [ ]:
docker exec -it sqlite sh

/var/lib/sqlite # ls
init_db.sh  sqlite.db

sqlite3 sqlite.db
sqlite>

sqlite> .tables
test

Si obtenemos un listado correcto de tablas, la base de datos está funcionando bien.

Esta vez vemos el scrip que crea la base de datos al levnatar el contenedor y una lista llamada 'test' de prueba.

 - ### Test: Verificar persistencia de datos
 
Insertar un dato en la base de datos y reiniciar el contenedor.



In [ ]:
CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO test (name) VALUES ('test_entry');

Si ya tenemos una tabla llamada test y nuestro SQLite funciona OK, nos debe de lanzar este error:

In [ ]:
Parse error: table test already exists
  CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT); INSERT INTO test (name)
               ^--- error here

Por lo tanto cambiaremos nuestro comando para que nos cree la tabla si no existe y que inserte un dato:

In [ ]:
CREATE TABLE IF NOT EXISTS test (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO test (name) VALUES ('test_entry');

Para ver los datos que hemos insertado ejecutamos este comando:

In [ ]:
SELECT * FROM test;

sqlite> SELECT * FROM test;
1|test_entry
sqlite> .exit
/var/lib/sqlite # exit

Tumbamos y volvemos a levantar el contenedor. Realizamos de nuevo los pasos y el dato debe de serguir en la tabla.

Ahora podemos borrar la tabla con:

In [ ]:
DROP TABLE test;

- ## b. Servicio Backend (Node.js API):

**Test: Verificar que la API está en ejecución**

- Comando:

    ```yaml
    curl http://localhost:3000/
    ```
- Acción: Realizar una petición GET al endpoint base del backend.
- Resultado esperado: La API debería devolver una respuesta válida, como un mensaje de bienvenida o un JSON con información básica.

    ```yaml
    ¡Hola, mundo desde Node.js!% 
    ```

**Test: Conexión con SQLite**  <font color="red">**(ERROR - Solucionado)**</font>

- Comando:

    ```yaml
    curl http://localhost:3000/api/test_db
    ```

- Nos devuelve un ERROR:
    ```yaml
    <!DOCTYPE html>
    <html lang="en">
    <head>
    <meta charset="utf-8">
    <title>Error</title>
    </head>
    <body>
    <pre>Cannot GET /api/test_db</pre>
    </body>
    </html>
    ```
- Acción: Endpoint que realice una consulta simple a SQLite, como obtener una lista de datos.
- Resultado esperado: Deberías recibir un JSON con los datos almacenados en la base de datos SQLite.

**Test: Verificar persistencia de datos del backend**

- Comando: Insertar un dato en el sistema mediante un endpoint POST y luego reiniciar el contenedor.

In [ ]:
curl -X curl -X POST http://localhost:3000/api/create -d '{"username":"test_user", "email":"test@example.com", "password":"123456"}' -H "Content-Type: application/json"

- Verificar con un GET

In [ ]:
curl http://localhost:3000/api/items

- Resultado esperado: El dato debería persistir después de reiniciar.

- Reconstruir el contenedor backend: Después de realizar estos cambios, necesitas reconstruir tu contenedor para aplicar los cambios en el código.

In [ ]:
make logs_service SERVICE=backend

app  | 
app  | > backend@1.0.0 start
app  | > node app.js
app  | 
app  | Servidor escuchando en http://localhost:3000
app  | Base de datos inicializada correctamente

- Probar las rutas:
    - Usar curl para probar las rutas y asegurarnos de que las consultas están funcionando correctamente. Por ejemplo:

- Obtener todos los usuarios:
- Obtener todos los juegos:
- Probar la conexión a la base de datos:

In [ ]:
curl http://localhost:3000/api/users
[]% 

curl http://localhost:3000/api/games
[]%

curl http://localhost:3000/api/test_db

Conexión a la base de datos exitosa%  

- ## c. Servicio PHP:

**Test: Verificar que PHP está en ejecución** <font color="red">**(ERROR)**</font>

 - Comando:
    ```yaml
    curl http://localhost:8080/
    ```
- Nos devuelve un error:
    ```yaml
    curl: (56) Recv failure: Connection reset by peer
    ```
- Acción: Realizar una petición GET al servidor PHP.
- Resultado esperado: Debería devolver una página PHP o un mensaje de bienvenida si está correctamente configurado.

In [ ]:
curl http://localhost:8080/

**Test: Conexión con Backend**

- Comando: Crear un formulario en PHP que haga una petición al backend (Node.js).
    ```yaml
    <?php
    $response = file_get_contents('http://backend:3000/api/test');
    echo $response;
    ?>
    ```
-  Acción: Acceder a esta página desde el navegador.
- Resultado esperado: El servidor PHP debería devolver el resultado de la API del backend.


In [ ]:
<?php
$response = file_get_contents('http://backend:3000/api/test');
echo $response;
?>